In [1035]:
import os
import sys

notebook_dir = os.getcwd()
target_folder = os.path.abspath(os.path.join(notebook_dir, ".."))
if target_folder not in sys.path:
    sys.path.append(target_folder)

import cards
import random

random.seed(43)

def random_seven(deck:list):
    board = random.sample(deck, 7)
    return board 

RANDOM = random_seven(cards.DECK)
SF   = ('9d','8d','7d','6d','5d','Ah','Kc')
TRAP = ('9s','7s','5s','3s','2s','8h','6d')
WHEEL= ('As','5d','4c','3h','2s','9d','Kc')
BOAT = ('As','Ah','Ad','Ks','Kh','Kd','Qs')
PAIR = ('Jd','Js','Ks','9h','7c','5d','2c')
TWO_PAIR = ('Jd','Js','Ks','Kh','7c','5d','2c')
QUADS = ('Jd','Js','Jh','Jc','7c','5d','2c')
TRIPS = ('Jd','Js','Th','Jc','7c','5d','2c')

In [1036]:
def convert_ranks(ranks:dict):
    mapped_ranks = {}
    for rank , count in ranks.items():
        value = cards.CARD_VALUE[rank]
        mapped_ranks[value] = count
    
    return mapped_ranks


def count_ranks_suits(seven:tuple):
    ranks = {}
    suits = {}
    for card in seven:
        ranks[card[0]] = ranks.get(card[0], 0) + 1
        suits[card[1]] = suits.get(card[1], 0) + 1
    
    mapped_ranks = convert_ranks(ranks)

    return mapped_ranks , suits

ranks, suits = count_ranks_suits(BOAT)
print(f'Suits: {suits}')
print(f'Ranks: {ranks}')

Suits: {'s': 3, 'h': 2, 'd': 2}
Ranks: {12: 3, 11: 3, 10: 1}


In [1037]:
def find_groups(ranks:dict):
    group_counts = {}
    for rank, count in sorted(ranks.items(), reverse=True):
        group_counts.setdefault(count , []).append(rank)
    
    return group_counts

find_groups(ranks)

{3: [12, 11], 1: [10]}

In [1038]:
def find_flush_suit(suits:dict):
    flush_suit = None
    for suit , count in suits.items():
        if count >= 5:
            flush_suit = suit
    
    return flush_suit

flush_suit = find_flush_suit(suits)

In [1039]:
def find_flush_cards(seven, suit):
    flush_cards = []
    for card in seven:
        if card[1] == suit:
            flush_cards.append(card)
    
    flush_cards_tuple = tuple(flush_cards)
    
    return flush_cards_tuple

flush_cards = find_flush_cards(BOAT, flush_suit)

flush_cards

()

In [1040]:
def find_straight(ranks):
    sorted_ranks = sorted(list(set(ranks)))
    straight_high = None
    consecutive_count = 1
    for i in range(len(sorted_ranks) - 1):
        if sorted_ranks[i] == sorted_ranks[i + 1] - 1:
            consecutive_count += 1
            if consecutive_count >= 5:
                straight_high = sorted_ranks[i + 1]
        else:
            consecutive_count = 1
    
    if straight_high is None:
        if {12,0,1,2,3}.issubset(set(ranks)):
            straight_high = 3

    return straight_high 

find_straight(ranks)

In [1041]:
def find_straight_flush(flush_cards):
    flush_ranks, flush_suits = count_ranks_suits(flush_cards)
    flush_straight_high = find_straight(flush_ranks)
    return flush_straight_high

find_straight_flush(flush_cards)


In [1042]:
def find_kickers(spent, groups):
    flat_list = []
    for i in groups:
        for value in groups[i]:
            flat_list.append(value)
    
    all_else = []
    for i in flat_list:
        if i in spent:
            pass
        else:
            all_else.append(i)

    return sorted(all_else, reverse= True)

In [1058]:
def evaluate_hand(seven):
    ranks, suits = count_ranks_suits(seven)
    groups = find_groups(ranks)
    flush_suit = find_flush_suit(suits)
    flush_cards = None
    sf_high = None
    
    if flush_suit is not None:
        flush_cards = find_flush_cards(seven, flush_suit)
        sf_high = find_straight_flush(flush_cards)
    
    straight_high = find_straight(ranks)
    
    pair = None
    trips = None
    pairs_list = groups.get(2,[])
    trips_list = groups.get(3, [])
    
    candidates = []
    
    if len(trips_list) >= 1:
        trips = groups[3][0]
        if len(trips_list) > 1:
            for i in trips_list:
                if i == trips:
                    pass
                else:
                    candidates.append(i)
        else:
            for i in pairs_list:
                candidates.append(i)
        if len(candidates) >= 1 :
            pair = max(candidates)


    if sf_high is not None:
        return (9, sf_high)
    elif groups.get(4) is not None:
        quads = groups[4][0]
        quad_list = [quads]
        kickers = find_kickers(quad_list, groups)
        kicker = kickers[0]
        return (8, quads, kicker)
    elif trips is not None and pair is not None:
        return (7, trips, pair)
    elif flush_suit is not None:
        flush_values = []
        for i in ranks:
            flush_values.append(i)
        return (6, flush_values[0], flush_values[1],flush_values[2], flush_values[3], flush_values[4])
    elif straight_high is not None:
        return (5, straight_high)
    elif trips is not None:
        kickers = find_kickers(trips_list, groups)
        return (4, trips, kickers[0], kickers[1])
    elif len(pairs_list) >= 2:
        high_pair = pairs_list[0]
        low_pair = pairs_list[1]
        kickers = find_kickers(pairs_list,groups)
        return (3, high_pair, low_pair, kickers[0])
    elif pairs_list == 1:
        pair = pairs_list[0]
        kickers = find_kickers(pairs_list, groups)
        return (2, pair, kickers[0], kickers[1], kickers[2])
    else:
        high_card = groups[1][0]
        k1 = groups[1][1]
        k2 = groups[1][2]
        k3 = groups[1][3]
        k4 = groups[1][4]
        return (1, high_card, k1, k2,k3,k4)
    

In [1059]:
evaluate_hand(RANDOM)

(1, 12, 11, 7, 5, 4)

In [1060]:
tp_suits, tp_ranks = count_suits_ranks(RANDOM)
tp_groups = find_groups(tp_ranks)

tp_groups

{1: [12, 11, 7, 5, 4, 2, 0]}